## Set path

In [ ]:
# Import data saving utilities (saveList, loadList) for HDF5 file operations
from utilities.saving import saveList, loadList

In [ ]:
# Import required libraries and modules
%matplotlib widget
import numpy as np
import os
import time
import datetime
import cv2
from ipywidgets import interact,IntSlider

In [ ]:
# Get fish birth month from user input for age calculation
m_birth = int(input("Enter m birth: ")) 

In [ ]:
# Get fish birth day from user input for age calculation
d_birth = int(input("Enter d birth: ")) 

In [ ]:
# Calculate fish age in days and create experiment directory with date-based naming
# Get current date
today = datetime.date.today()
# Set fish birth date
someday = datetime.date(2026, m_birth, d_birth)
# Calculate age difference in days
diff = today - someday 
# Extract age as integer number of days
age=diff.days
print('age', age)
# Define directory path for data storage
# Root folder for raw data (set DATA_ROOT, defaults to D:\data_freely_swimming on the lab PC)
path = os.environ.get('DATA_ROOT', r'D:\data_freely_swimming')
# Construct experiment directory path with date and fish ID
dir_experiment_day = os.path.join(path,'fs_'+ str(age)+'dpf_'+datetime.datetime.now().strftime("%d") + datetime.datetime.now().strftime("%b") + datetime.datetime.now().strftime("%y"))
# Check if path exists
if not os.path.exists(dir_experiment_day):
    print('Directory created properly, path: ',dir_experiment_day)
    # Create directory path or directory
    os.mkdir(dir_experiment_day)
else:
    print('ERROR: directory already exists.')


In [ ]:
# Get fish number from user and create individual fish experiment directory
fish_number = (input("Enter fish number: ")) 
# Define directory path for data storage
dir_experiment_fish = os.path.join(dir_experiment_day,'fish_'+ str(fish_number))
# Create directory path or directory
# Handle directory/file existence
if not os.path.exists(os.path.join(path,dir_experiment_fish)):
    print('Directory created properly, path: ',dir_experiment_fish)
    # Create directory path or directory
    os.mkdir(os.path.join(path,dir_experiment_fish))
else:
    print('ERROR: use a new fish_number, not', fish_number)
# Get current date
date_in_arena = time.asctime()
print(date_in_arena)

In [ ]:
# Get user input for experiment parameters
sb = int(input("Enter sb: ")) 


# Set projector

In [ ]:
# Configure PsychoPy window for projector or second screen with calibration parameters
from psychopy import visual,event,clock,monitors,core
from PIL import Image
projector=False
# Check projector configuration
# Configure display device (projector vs second screen)
if projector:
    x_center_motor = 195.0 #calibration_26Mar24
    y_center_motor = 231.0
    proj_width=84
    # Initialize PsychoPy monitor and window for stimulus display
    mon=monitors.Monitor('projector')
    mon.setWidth(proj_width) 
    mon.setSizePix([1920,1080])
    mon.save()  
    screen_id=1 
else: #second screen
    x_center_motor = 213.0 #calibration_06June24
    y_center_motor = 248.0
    proj_width=70
    # Initialize PsychoPy monitor and window for stimulus display
    mon = monitors.Monitor('second_screen')
    mon.setWidth(proj_width)  # Adjust the physical width of the monitor in cm
    mon.setSizePix([2560, 1440])  # Resolution of the second screen
    mon.save()
    screen_id = 0
# Create a window on the projector/second screen
win = visual.Window(
    size=mon.getSizePix(),
    monitor=mon,
    fullscr=True,
    screen=screen_id,
    units='cm',
    color='white')


In [ ]:
# Main loop to keep the window open and listen for a key press to exit
while True:
    win.flip()  # Update the window
    keys = event.getKeys()
    if 's' in keys:  # Check if the 's' key was pressed
        break
    event.clearEvents()
win.close()  # Close the window


# Libreries

In [ ]:
# Set experimental parameters and configuration variables
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'   


In [ ]:
# Load experimental data from file
%load_ext autoreload
%autoreload 2

In [ ]:
# Import required libraries and modules
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Import required libraries and modules
import matplotlib.pyplot as plt
import json
import math
import keyboard
import re #regex
import threading
import queue
from skimage.io import imsave
from skimage.transform import downscale_local_mean
import matplotlib.patches as patches
from scipy import ndimage

import h5py

from scipy import ndimage
from scipy.signal import find_peaks, peak_widths

In [ ]:
# Import data saving utilities (saveList, loadList) for HDF5 file operations
from utilities.camera_PointGrey import _initialize, _cleanup, capture
from utilities.preprocessing import preprocess_im, bw_image, noise_removal, area_contours, bw_image_simple, background_sub, background_sub2, background_sub_uint8
from utilities.Shapeoko import ShapeokoStatus, ShapeokoWrite, ShapeokoStart, ShapeokoHoming, ShapeokoKill, ShapeokoMove, ShapeokoBoundary, ShapeokoBoundary2, ShapeokoFollow_simple, ShapeokoFollow_osc, ShapeokoFollow_osc2, ShapeokoFollow_osc3,ShapeokoFollow_tabs, ShapeokoFollow_TEST, ShapeokoFollow_TEST2
from utilities.tracking import centroid, centroid2, get_ellipse, img_draw_ellipse, fish_position, find_nearest
from utilities.live_kinematic_parameters import Fish_live
from utilities.projector_and_Shapeoko import manual_mode_projector, manual_mode_screen
from utilities.psychopy_projection import closeloop_bg_screen, stones_bg_screen_4moving_patter, stones_bg_proj, stones_bg_proj_RandomAngle, stones_bg_screen, stones_bg_screen_framed
from utilities.saving import cropImage
from utilities.mask import mask_creation, take_mask_2, mask_status_report, mask_status_report_2, visualize_mask
from utilities.cleaning_edge import clear_res_border 
from utilities.reading import h5tree_view, sorted_alphanumeric, load_mp4_as_numpy
from utilities.my_fit import fit_line, build_line, build_line_inv
from utilities.sanity_test import send_sms, calculate_distance, check_motor_movement
from utilities.monitor_and_Shapeoko import manual_monitor_red
from utilities.tracking_yolo import crop_around_rectangle, crop_around_point
from utilities.tracking_head_dir import check_direction_difference

In [ ]:
# Configure PyTorch to use CUDA GPU if available, otherwise use CPU
import torch
# Check if CUDA GPU is available
if torch.cuda.is_available():
    # PyTorch device (CPU/GPU)
    device = torch.device("cuda")
else:
    # PyTorch device (CPU/GPU)
    device = torch.device("cpu")
    print("using", device, "device")


In [ ]:
# Import required libraries and modules
import torchvision
import torchvision.transforms as transforms


In [ ]:
# PIL->tensor
tr_PILtoTens = transforms.Compose([ transforms.PILToTensor() ]) 

# tensor->PIL
tr_TensToPIL = transforms.ToPILImage()

In [ ]:
# Import YOLO model from Ultralytics for deep learning-based object detection
from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()
from ultralytics import YOLO

from matplotlib.patches import Rectangle

# Set general parameters

In [ ]:
# Process background image for subtraction-based tracking
gaussian_sigma=1 #standard deviation in pixels of the gaussian used to filter the images
downsampling=2
bb=5 
name_img='bg-grayscale.jpg' #background images for freely swiming
to_number = os.environ.get('ALERT_PHONE_NUMBER')  # optional SMS alerts, see utilities/sanity_test.py
stones_bg_frame = True


# Camera

In [ ]:
# Initialize Point Grey camera system, configure resolution, offset, and frame rate (250 fps)
from rotpy.system import SpinSystem
from rotpy.camera import CameraList
# Initialize Point Grey camera system
system = SpinSystem()
# Camera variable initialization
# Initialize Point Grey camera system
cameras = CameraList.create_from_system(system, True, True)
# Camera variable initialization
camera = cameras.create_camera_by_index(0)
camera.init_cam()
camera.camera_nodes.Width.set_node_value(608)
camera.camera_nodes.Height.set_node_value(600)
camera.camera_nodes.OffsetX.set_node_value(376)#336
camera.camera_nodes.OffsetY.set_node_value(320)#212
fps=250
camera.camera_nodes.AcquisitionFrameRate.set_node_value(fps,False)
print(camera.camera_nodes.AcquisitionFrameRate.get_node_value())
camera.deinit_cam()

In [ ]:
# Initialize camera, capture test image, and cleanup for calibration
_initialize(camera)
# Capture image from camera
[im_raw,timestamp]=capture(camera)
# Initialize or cleanup camera
_cleanup(camera)
img_crop=cropImage(im_raw, 300 , 300, 500)
img_crop_ds=downscale_local_mean(img_crop,downsampling)
x1, y1 = 60,86
x2, y2 = 203,y1
length_pix = np.sqrt((x1-x2)**2+(y1-y2)**2) # pixels
lengths_mm = 16# mm
mm_per_pix=round(lengths_mm/length_pix,3) #in the processed image
print(mm_per_pix)
%matplotlib widget
# Create new figure for plotting
plt.figure(figsize=[5,5])
# Display the plot
plt.imshow(img_crop_ds,origin='lower')
# Plot data points
plt.plot(x1, y1,'ro')
plt.plot(x2, y2,'ro')
plt.xlabel('x [pixel]')
plt.ylabel('y [pixel]')
# Display the plot
plt.show()


# Shapeoko

In [ ]:
# Initialize serial connection to Shapeoko CNC machine
try:
    # Serial connection to Shapeoko
    ser=ShapeokoStart()
# Handle exceptions/errors
except:
    # Execute Shapeoko motor command
    ShapeokoKill(ser)
    # Serial connection to Shapeoko
    ser=ShapeokoStart()


In [ ]:
# Perform homing sequence to set Shapeoko to known reference position
ShapeokoHoming(ser)


In [ ]:
# Initialize and configure Shapeoko CNC machine for motor control
ShapeokoWrite(ser, '?', True)


In [ ]:
# Initialize and configure Shapeoko CNC machine for motor control
ShapeokoWrite(ser, '$G', True) #to check relative/absolute


# Background

In [ ]:
#absolute position mode
print('Absolute mode')
ser.write(b'G90\n')
tmp = 'G0X+' + str(200) + 'Y+' + str(200) + '\n'
ser.write(str.encode(tmp)) #centering

In [ ]:
# Initialize camera, capture test image, and cleanup for calibration
ser.write(b'G91\n') 
bg_tot=[]
cum_x,cum_y = 0,0
dl = 10 
sign = +1
# Loop while condition is true
# Iterate over data collection
while cum_x < 10:
    # Loop while condition is true
    # Iterate over data collection
    while cum_y < 10:
        # Execute Shapeoko motor command
        ShapeokoMove(ser, 0, sign*dl,1000)
        time.sleep(0.1)
        cum_y = cum_y + dl
        if camera.is_streaming():
            # Initialize or cleanup camera
            _cleanup(camera)
        # Initialize or cleanup camera
        _initialize(camera)
        # Capture image from camera
        [background_raw,timestamp]=capture(camera)
        # Initialize or cleanup camera
        _cleanup(camera)
        bg_raw_crop=cropImage(background_raw, 300 , 300, 500)
        bg_raw_crop_ds=downscale_local_mean(bg_raw_crop,downsampling)
        bg_tot.append(bg_raw_crop_ds)       
    cum_y = 0
    sign=-sign  
    # Execute Shapeoko motor command
    ShapeokoMove(ser, dl, 0)
    time.sleep(0.1)
    cum_x = cum_x + dl  
bg_mean=np.array(bg_tot).mean(axis=0)
# Create directory path or directory
np.save(os.path.join(dir_experiment_day,'bg_mean.npy'),bg_mean) 
img_bw=bg_mean

In [ ]:
# Process background image for subtraction-based tracking
plt.figure(dpi=50)
# Display the plot
plt.imshow(bg_mean,origin='lower')
plt.colorbar()
plt.xlabel('x [pixel]')
plt.ylabel('y [pixel]')
# Display the plot
plt.show()


In [ ]:
# Perform spatial calibration to convert pixels to physical units
xROIcenter=bg_mean.shape[0]/2
yROIcenter=bg_mean.shape[1]/2
w_img=bg_mean.shape[0]
h_img=bg_mean.shape[1]
print('ROI (pix)',w_img,h_img)
print('ROIcenter (pix)',xROIcenter,yROIcenter)
xROI_mm=round(w_img*mm_per_pix,2)
yROI_mm=round(h_img*mm_per_pix,2)
print('ROI (mm)', xROI_mm,yROI_mm)


# Define motor limit, agar edges

In [ ]:
# Set experimental parameters and configuration variables
ridefine_edge=False


In [ ]:
# Set experimental parameters and configuration variables
mot_xA,mot_xD=15,15
mot_yA,mot_yB=35,35
mot_yC,mot_yD=412,412
mot_xB,mot_xC=400,400 #not more then 400


In [ ]:
if ridefine_edge:
    ser.write(b'G90')
    ShapeokoMove(ser,mot_xA,mot_yA)
    time.sleep(1) 

In [ ]:
_initialize(camera)
[im_raw,timestamp]=capture(camera)
_cleanup(camera)

img_crop=cropImage(im_raw, 300 , 300, 500)
img_crop_ds=downscale_local_mean(img_crop,downsampling)
xA,yA=150,120 #xA,yA=70,55
%matplotlib widget
plt.figure(figsize=[2,2])
plt.imshow(img_crop_ds,origin='lower')
plt.plot(xA, yA,'ro')
plt.xlabel('x [pixel]')
plt.ylabel('y [pixel]')

agar_xA,agar_yA=mot_xA+xA*mm_per_pix,mot_yA+yA*mm_per_pix
print(mot_xA,mot_yA)
print(agar_xA,agar_yA)

plt.show()

In [ ]:
if ridefine_edge:
    ser.write(b'G90')
    ShapeokoMove(ser,mot_xB,mot_yB)
    time.sleep(1) 

In [ ]:
_initialize(camera)
[im_raw,timestamp]=capture(camera)
_cleanup(camera)
img_crop=cropImage(im_raw, 300 , 300, 500)
img_crop_ds=downscale_local_mean(img_crop,downsampling)

xB,yB=200,120#217,40
%matplotlib widget
plt.figure(figsize=[2,2])
plt.imshow(img_crop_ds,origin='lower')
plt.plot(xB, yB,'ro')
plt.xlabel('x [pixel]')
plt.ylabel('y [pixel]')
agar_xB,agar_yB=mot_xB+(xB-w_img)*mm_per_pix,mot_yB+yB*mm_per_pix
print(mot_xB,mot_yB)
print(agar_xB,agar_yB)

plt.show()

In [ ]:
if ridefine_edge:
    ser.write(b'G90')
    ShapeokoMove(ser,mot_xC,mot_yC)
    time.sleep(1) 

In [ ]:
_initialize(camera)
[im_raw,timestamp]=capture(camera)
_cleanup(camera)

img_crop=cropImage(im_raw, 300 , 300, 500)
img_crop_ds=downscale_local_mean(img_crop,downsampling)
xC,yC=210,150#220,171

%matplotlib widget
plt.figure(figsize=[2,2])
plt.imshow(img_crop_ds,origin='lower')
plt.plot(xC, yC,'ro')
plt.xlabel('x [pixel]')
plt.ylabel('y [pixel]')
agar_xC,agar_yC=mot_xC+(xC-w_img)*mm_per_pix ,mot_yC+(yC-h_img)*mm_per_pix
print(mot_xC,mot_yC)
print(agar_xC,agar_yC)
plt.show()

In [ ]:
if ridefine_edge:
    ser.write(b'G90')
    ShapeokoMove(ser,mot_xD,mot_yD)
    time.sleep(1) 

In [ ]:
_initialize(camera)
[im_raw,timestamp]=capture(camera)
_cleanup(camera)
img_crop=cropImage(im_raw, 300 , 300, 500)
img_crop_ds=downscale_local_mean(img_crop,downsampling)
xD,yD=160,140 #70,184

%matplotlib widget
plt.figure(figsize=[2,2])
plt.imshow(img_crop_ds,origin='lower')
plt.plot(xD, yD,'ro')
plt.xlabel('x [pixel]')
plt.ylabel('y [pixel]')
agar_xD,agar_yD=mot_xD+(xD)*mm_per_pix,mot_yD+(yD-h_img)*mm_per_pix
print(mot_xD,mot_yD)
print(agar_xD,agar_yD)
plt.show()

In [ ]:
m1,b1=fit_line(agar_xA,agar_yA, agar_xB,agar_yB)
[ab_aux,ab_fit]=build_line(m1,b1,agar_xA,agar_xB)
m2,b2=fit_line(agar_xB,agar_yB, agar_xC,agar_yC)
[bc_aux,bc_fit]=build_line(m2,b2,agar_xB,agar_xC)
m3,b3=fit_line(agar_xC,agar_yC, agar_xD,agar_yD)
[cd_aux,cd_fit]=build_line(m3,b3,agar_xC,agar_xD)
m4,b4=fit_line(agar_xD,agar_yD, agar_xA,agar_yA)
[da_aux,da_fit]=build_line(m4,b4,agar_xA,agar_xD)

agar_abs=np.array([agar_xA,agar_yA,agar_xB,agar_yB,agar_xC,agar_yC,agar_xD,agar_yD])
motor_abs=np.array([mot_xA,mot_yA,mot_xB,mot_yB,mot_xC,mot_yC,mot_xD,mot_yD])

#define tab_y and tab_x
mot_y=np.arange(round(agar_yA), round(agar_yC)+1, 1)
tab_y=np.empty((mot_y.shape[0], 3))
for i in range(len(tab_y)):
    _y=mot_y[i]    
    start_s=np.max([0,_y-(yROI_mm/2)])
    end_s=np.min([len(tab_y),_y+(yROI_mm/2)])
    if start_s>end_s:
        start_s=np.min([start_s,end_s])
        end_s=np.max([start_s,end_s])
    [x_aux1,bc_slice]=build_line_inv(m2,b2,start_s,end_s)
    [x_aux2,da_slice]=build_line_inv(m4,b4,start_s,end_s)
    min_x_abs=np.mean(x_aux2)
    max_x_abs=np.mean(x_aux1)
    tab_y[i]=[int(mot_y[i]),min_x_abs,max_x_abs] #y_mot_abs, min_x_abs, max_x_abs

mot_x=np.arange(round(agar_xA), round(agar_xB)+1, 1)
tab_x=np.empty((mot_x.shape[0], 3))
for i in range(len(tab_x)):
    _x=mot_x[i]    
    start_s=np.max([0,_x-(xROI_mm/2)])
    end_s=np.min([len(tab_x),_x+(xROI_mm/2)])
    if start_s>end_s:
        start_s=np.min([start_s,end_s])
        end_s=np.max([start_s,end_s])
    [x_aux1,ab_slice]=build_line(m1,b1,start_s,end_s)
    [x_aux2,cd_slice]=build_line(m3,b3,start_s,end_s)
    min_y_abs=np.mean(ab_slice)
    max_y_abs=np.mean(cd_slice)
    tab_x[i]=[int(mot_x[i]),min_y_abs,max_y_abs] #x_mot_abs, min_y_abs, max_y_abs

%matplotlib widget
plt.figure()
plt.plot([agar_xA, agar_xB, agar_xC, agar_xD, agar_xA],[agar_yA, agar_yB, agar_yC, agar_yD, agar_yA],'ko-', label='agar edge')
plt.plot([mot_xA, mot_xB, mot_xC, mot_xD, mot_xA],[mot_yA, mot_yB, mot_yC, mot_yD, mot_yA],'r-', label='mot edge')
plt.plot(tab_y[:,1],tab_y[:,0],'.')
plt.plot(tab_y[:,2],tab_y[:,0],'m.')
plt.plot(tab_x[:,0],tab_x[:,1],'b.')
plt.plot(tab_x[:,0],tab_x[:,2],'g.')
test_x,test_y=tab_x[38,0],tab_x[38,1]
plt.plot(test_x,test_y,'o')

plt.title('Check tab-mask accuracy')
plt.xlabel('x (mm)')
plt.ylabel('y (mm)')
plt.xlim(-2,450)
plt.ylim(0,450)
plt.legend()
plt.show()

# Threds:

## camera

In [ ]:
# Process background image for subtraction-based tracking
# Define function for camera operations
def run_camera_simple(lock_fromcamera, fromcamera, stop_camera):
    first_loop = True
    real_time_exp=0
    frameid=0
    while not stop_camera.is_set(): #loop until stop_camera eventthreding

        [img_raw,timestamp]=capture(camera) 
        cameratimestamp.append(timestamp)
    
        if first_loop:
            first_loop=False
        else:
            real_time_exp=real_time_exp+(cameratimestamp[-1]-cameratimestamp[-2]) #set the clock     
    
        img_crop=cropImage(img_raw, 300 , 300, 500)
        img_crop_ds=downscale_local_mean(img_crop,downsampling)
        img_crop_ds_bg=background_sub(img_crop_ds,bg_mean) 
        
        lock_fromcamera.acquire() 
        fromcamera[0]=real_time_exp
        fromcamera[1]=img_crop_ds_bg
        fromcamera[2]=1
        lock_fromcamera.release()
        
        cameratimes.append(time.perf_counter()-t0) #clock
        # put image in a queue to be saved  
        queue_images.put([img_crop_ds_bg,frameid]) 
        frameid=frameid+1

## tracking

In [ ]:
# Load pre-trained YOLO model for fish detection and tracking
model = YOLO('yolo8m_live_tracking.pt')

In [ ]:
def weighed_conf(conf_t,th_conf):
    return -((1-np.exp(conf_t/th_conf))/np.exp(conf_t/th_conf))
             
test_conf=np.linspace(0,1,100)
test_f=[]
th_conf=0.1
for conf_t in test_conf:
    test_f.append(weighed_conf(conf_t,th_conf))
plt.figure(dpi=50)
plt.plot(test_conf,test_f)
plt.plot(test_conf,test_conf)
plt.grid()
plt.show()    

In [ ]:
# Define function for fish tracking
deb_crop=[]
pos_fish_yolo=[]
pos_kine=[]
def run_tracking(lock_fromcamera, fromcamera, lock_fromtracking, fromtracking, stop_tracking, lock_plotting, data_plotting, lock_pos_motor, motor_live):
    first_loop = True
    frame_fish_lost=0
    fish_lost=False
    global failed
    frameid=0
    dt_plot=0

    while not stop_tracking.is_set(): #loop until stop_camera event
     
        if first_loop:
            tau0=time.perf_counter()
            first_loop=False
            if flag_plot:
                trd_run_plotting.start()
            if flag_save:
                trd_save_images.start() 
            if flag_img_bw:
                trd_save_images_bw.start() 
                
            xfish_t, yfish_t = w_img/2, w_img/2
            x_kine, y_kine = w_img/2, w_img/2
            x_yolo , y_yolo = w_img/2, w_img/2
            pos_fish_yolo.append([x_yolo , y_yolo]) #pix
            pos_kine.append([x_kine, y_kine])
            pos.append([xfish_t , yfish_t])
            head_dir.append([0,0])
            
        else: #> first loop
            lock_pos_motor.acquire() #img from camera
            flag_motor=motor_live[3]
            if flag_motor==1:
                x_motor_live=motor_live[0]
                y_motor_live=motor_live[1]
                edge=motor_live[2]
                motor_live[3]=0
                motor_pos_live.append([x_motor_live , y_motor_live]) #mm
                motor_time_live.append(time.perf_counter()-t0)
                tau=time.perf_counter()-tau0
                lock_pos_motor.release()  
            else:
                tau0=time.perf_counter()
                lock_pos_motor.release()
            
            flag=fromcamera[2]
            lock_fromcamera.acquire() #img from camera
            if flag==1:
                img_camera_now=fromcamera[1]
                fromcamera[2]=0
                frameid=frameid+1
                lock_fromcamera.release()  
                
                ######====> position extracted from YOLO OBJECT DETECTION
                start = time.time()
                img_yolo=img_camera_now#img_camera_now*img_bw
                #downscale -> yolo8m on imsize=128x128
                img_ds=downscale_local_mean(img_yolo,2)
                img_ds=np.uint8(img_ds)
                #from numpy to PIL
                rgb_image = np.stack([img_ds]*3,axis=-1)
                pil_image=Image.fromarray(rgb_image, 'RGB')
                # PIL->tensor
                img_tensor = tr_PILtoTens(pil_image) 
                # tensor TO gpu
                img_tensor_gpu = img_tensor.to(device)
                torch.cuda.synchronize()
                # tensor->PIL
                img_ds_gpu = tr_TensToPIL(img_tensor_gpu)
                results = model(img_ds_gpu, show=False, conf=0.01,verbose=False)
                torch.cuda.synchronize()
                
                for result in results:
                    boxes = result.boxes.cpu().numpy()
                    xyxys = boxes.xyxy
                conf_level.append(result.boxes.conf.tolist())
                
                if len(xyxys)>1: #more then one obj detected
                    #print('🔴 YOLO detected more then one obj')
                    x_obj_yolo=[]
                    y_obj_yolo=[]
                    for obj in range(len(xyxys)): #more then one obj detected
                        x_obj_yolo.append(((xyxys[obj][2]-xyxys[obj][0])/2+xyxys[obj][0])*2)
                        y_obj_yolo.append(((xyxys[obj][3]-xyxys[obj][1])/2+xyxys[obj][1])*2) 
                    dist=np.sqrt(( np.array(x_obj_yolo)-pos[-1][0] )**2 + ( np.array(y_obj_yolo)-pos[-1][1] )**2)
                    id_fish=np.argmin(dist) #the closest respect pos_fish(t-1)  
                    
                    if not np.isnan(xyxys[id_fish]).any():
                        x_yolo , y_yolo = x_obj_yolo[id_fish],y_obj_yolo[id_fish]
                        conf_t = conf_level[-1][id_fish]
                    else:
                        x_yolo , y_yolo = pos[-1][0], pos[-1][1] 
                        conf_t = 0
                else:
                    if len(xyxys)==1:
                        #print('💚 YOLO detected one obj in', round((time.time()-start)*1000,0), '(ms)')
                        id_fish=0
                        if not np.isnan(xyxys[0]).any():            
                            x_yolo = ((xyxys[id_fish][2]-xyxys[id_fish][0])/2+xyxys[id_fish][0])*2
                            y_yolo = ((xyxys[id_fish][3]-xyxys[id_fish][1])/2+xyxys[id_fish][1])*2
                            conf_t = conf_level[-1][id_fish]
                        else:
                            #print('💀 YOLO returned nan')
                            x_yolo , y_yolo = pos[-1][0], pos[-1][1]
                            conf_t = 0
                    else:
                        #print('❌ YOLO detected nothing')
                        x_yolo , y_yolo = pos[-1][0], pos[-1][1]   
                        xyxys = [np.array([np.nan, np.nan, np.nan, np.nan])]
                        conf_t = 0
                        
                ######====> position extracted from ELLIPSE
                dt=1/20
                try:
                    img_crop=crop_around_point(img_camera_now, pos[-1])
                    ret, img_th = cv2.threshold(img_crop, 20, 1, cv2.THRESH_BINARY)
                    img_bw=noise_removal(img_th, 20)
                    E = get_ellipse(img_bw)
                    dx,dy = E.x2-E.x, E.y2-E.y
                    norm=np.sqrt(dx**2+dy**2)
                    head_dir.append([dx/norm, dy/norm])
                    debug_crop.append(img_crop)
                except:
                    conf_t = 1 #consider only yolo, no enough kinematic parameters
                    head_dir.append(head_dir[-1])
                    debug_crop.append(np.zeros((100,100)))
                tmp=((v_max_fish*dt)/mm_per_pix)*2
                x_kine, y_kine = max(0, min(pos[-1][0]+tmp*head_dir[-1][0], w_img)) , max(0, min(pos[-1][1]+tmp*head_dir[-1][1], w_img))  
                
                #Kalman filter : pos_fish_yolo and pos_fish_kine
                trackingtimes.append(time.perf_counter()-t0) #clock
                if trackingtimes[-1]>20 and edge and check_direction_difference(np.array(head_dir)):
                    xfish_t = weighed_conf(conf_t,th_conf)*(x_yolo-x_kine) + x_kine
                    yfish_t = weighed_conf(conf_t,th_conf)*(y_yolo-y_kine) + y_kine
                else:
                    xfish_t = x_yolo
                    yfish_t = y_yolo
         
                pos.append([xfish_t , yfish_t])
                pos_fish_yolo.append([x_yolo , y_yolo]) #pix
                pos_kine.append([x_kine, y_kine])

                if len(pos)>250*2:
                    if conf_t<0.03:
                        fish_lost=True
                    else:
                        fish_lost=False
                
                #print('END xfish_t,x_kine,x_yolo,conf_t=',xfish_t,x_kine,x_yolo,conf_t)#=======
                #copy info to comunicate online...
                lock_fromtracking.acquire() #...to the motor thread 
                fromtracking[0]=xfish_t
                fromtracking[1]=yfish_t
                fromtracking[2]=fish_lost
                fromtracking[3]=1
                lock_fromtracking.release()
    
                if flag_plot:
                    if trackingtimes[-1]-dt_plot>0.5:
                        lock_plotting.acquire() #...to the plotting thread
                        data_plotting[0]=img_camera_now
                        data_plotting[1]=np.array([xfish_t , yfish_t])
                        #data_plotting[1]=np.array([weighed_conf(conf_t,th_conf)*(x_yolo-x_kine) + x_kine , weighed_conf(conf_t,th_conf)*(y_yolo-y_kine) + y_kine])
                        data_plotting[2]=np.array([x_kine , y_kine]) #KINE
                        data_plotting[3]=np.array([x_yolo , y_yolo]) #YOLO
                        lock_plotting.release() 
                        dt_plot=trackingtimes[-1]
            else:
                lock_fromcamera.release()
                time.sleep(0.001)

## motor

In [ ]:
dt_motor=round( ((1/18)/60), 3) #min
# Calibrate motor based on fish age 
if age<10:
    v_max= 30 #mm/s
    v_max_fish=30
    time.sleep(0.01)
    ser.write(b'$120=100.000\n') #X-axis acceleration, mm/min^2
    time.sleep(0.01)
    ser.write(b'$121=100.000\n')
if age>=10 and age<25:
    v_max=400#50 #mm/s
    v_max_fish=50
    time.sleep(0.01)
    ser.write(b'$120=450.000\n') #X-axis acceleration, mm/min^2 #100.000
    time.sleep(0.01)
    ser.write(b'$121=450.000\n')
if age >=25:
    v_max= 600 #mm/s
    v_max_fish=100
    time.sleep(0.01)
    ser.write(b'$120=550.000\n') #X-axis acceleration, mm/min^2
    time.sleep(0.01)
    ser.write(b'$121=550.000\n')

print('age:',age, ',v_max:', v_max, 'mm/s')
R_max=round( ( bg_mean.shape[0]*mm_per_pix )/2 - ( bg_mean.shape[0]*mm_per_pix )/8  , 2 )
alpha=round( ( v_max/R_max ) * 60, 2) #1/min
print('alpha:',alpha)
#beta=round( 0.8*np.sqrt(4*alpha)*dt_motor ,3) 
beta = 0.1
print('beta:',beta)

def speed_osc(alpha, beta, r, v_t1):
    r=-r
    v_t2 = - alpha * r + (1-beta) * v_t1
    return round(v_t2, 0) #mm/min

In [ ]:
#Function that interacts with the motor.
dstep=[]
last_points=4000 #2400->  ~2min
def run_motor(stop_motor, tomotor, lock_tomotor, motor_live, lock_pos_motor):   
    v0 = v_max*60 #mm/min
    searching_step=0
    global failed
    #absolute position mode
    ser.write(b'G90\n')
    time.sleep(0.01)
    step_delay=0
    edge=False
    while not stop_motor.is_set(): #loop until stop_motor event
        dxr,dyr=0,0
        v_mms=0
        
        msg_status=ShapeokoWrite(ser,'?',False)    
        if 'WPos:' in msg_status:
            s0=re.split('[|]',msg_status) 
            wpos=re.split('[:,]',s0[1])  
            buffer=re.split('[:,]',s0[2])  
            motorpositions.append([float(wpos[1]), float(wpos[2])])
            bf=float(buffer[1])
            if s0[0]=='<Idle' and bf<35:
                step_delay+=1
            if s0[0]=='<Idle' and bf==35:
                step_delay=0
        else:
            motorpositions.append([np.nan, np.nan])
            bf = np.nan
        motortimes.append(time.perf_counter()-t0) #clock
        #if edge:
        #    bf_size=34
        #else:
        #    bf_size=33
        bf_size=34
        if bf>=bf_size: 
            lock_tomotor.acquire()
            flag=tomotor[3]
            if flag==1:
                x_centroid_now=tomotor[0]
                y_centroid_now=tomotor[1] 
                fish_lost=tomotor[2]
                tomotor[3]=0
                lock_tomotor.release()

            dx=round( (x_centroid_now - xROIcenter)*(mm_per_pix), 3)
            dy=round( (y_centroid_now - yROIcenter)*(mm_per_pix), 3)
            r=round( np.sqrt(dx**2+dy**2), 2)
            
            if fish_lost==False: 
                searching_step=0
                v = speed_osc(alpha, beta, r, v0)
                if r>0.1:
                    #edge=ShapeokoFollow_TEST(ser, motorpositions[-1][0], motorpositions[-1][1], mot_xA, mot_yB, mot_xB, mot_yD, tab_x, tab_y, dx, dy, v)
                    edge=ShapeokoFollow_TEST2(ser, motorpositions[-1][0], motorpositions[-1][1], mot_xA, mot_yB, mot_xB, mot_yD, tab_x, tab_y, dx, dy, v)
                    v0=v
            else:
                searching_step = searching_step + 1
                if searching_step<100:
                    if ~np.isnan(motorpositions[-1][0]) and ~np.isnan(motorpositions[-1][1]):
                        big_step_x,big_step_y=dx+(np.sign(dx)),dy+(np.sign(dy))
                        #edge=ShapeokoFollow_TEST(ser, motorpositions[-1][0], motorpositions[-1][1],mot_xA, mot_yB, mot_xB, mot_yD, tab_x, tab_y, big_step_x, big_step_y)
                        edge=ShapeokoFollow_TEST2(ser, motorpositions[-1][0], motorpositions[-1][1],mot_xA, mot_yB, mot_xB, mot_yD, tab_x, tab_y, big_step_x, big_step_y)
                    #print('Searching...',searching_step)
                else:
                    print('Fish definitly lost')
                    failed = True 
                    
        #if len(motorpositions)>last_points: 
        #    if not check_motor_movement(motorpositions,last_points):
        #        print('motor not moving from too long')
        #        failed = True 

        lock_pos_motor.acquire() 
        motor_live[0]=motorpositions[-1][0]
        motor_live[1]=motorpositions[-1][1]
        motor_live[2]=edge
        motor_live[3]=1
        lock_pos_motor.release() 

## plotting

In [ ]:
# Define function for data visualization
def run_plotting(lock_plotting, data_plotting, stop_plotting):   
    while not stop_plotting.is_set(): 
        lock_plotting.acquire()
        im_now=np.copy(data_plotting[0])
        pos_now=np.copy(data_plotting[1])
        kine_now=np.copy(data_plotting[2])
        yolo_now=np.copy(data_plotting[3])
        lock_plotting.release()
        
        fig.canvas.restore_region(bg)
        plot_live.set_data(im_now)
        plot_pos.set_xdata(pos_now[0])
        plot_pos.set_ydata(pos_now[1])
        plot_kine.set_xdata(kine_now[0])
        plot_kine.set_ydata(kine_now[1])
        plot_yolo.set_xdata(yolo_now[0])
        plot_yolo.set_ydata(yolo_now[1])
    
        ax.draw_artist(plot_live)
        ax.draw_artist(plot_pos)
        ax.draw_artist(plot_kine)
        ax.draw_artist(plot_yolo)

        fig.canvas.blit(fig.bbox)
        fig.canvas.flush_events()
        plottingtimes.append(time.perf_counter()-t0)
        time.sleep(0.001)

## saving

In [ ]:
# Set up experiment directory structure and file paths
video_mp4=True #you can choose to save the video in np.array or .mp4
if video_mp4:
    def run_saving_img(queue_images,stop_saving,dir_experiment_run):
        
        os.makedirs(os.path.join(dir_experiment_run, 'movie'), exist_ok=True)
    
        fps = 250.0
        codec = 'mp4v'
        fourcc = cv2.VideoWriter_fourcc(*codec)
        l_save_min = 1  # video split every ~2 minutes
        Delta_save = int(fps * l_save_min * 60)  # total frames per chunk
    
        movie_id = 0
        out = None
        frame_counter = 0
    
        while not (stop_saving.is_set() and queue_images.empty()):
            if not queue_images.empty():
                frame, frame_id = queue_images.get()
    
                # Init writer once we have the first frame
                if out is None:
                    if frame.ndim == 3:
                        frame_height, frame_width = frame.shape[:2]
                        is_color = True
                    else:
                        frame_height, frame_width = frame.shape
                        is_color = False
    
                    output_file = os.path.join(dir_experiment_run, 'movie', f'movie_{movie_id}.mp4')
                    out = cv2.VideoWriter(output_file, fourcc, fps, (frame_width, frame_height), isColor=is_color)
                    print(f"[INFO] Writing to {output_file}")
    
                # Make sure grayscale is 3-channel if needed
                if is_color and frame.ndim == 2:
                    frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
                elif not is_color and frame.ndim == 3:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
                out.write(frame)
                frame_counter += 1
    
                # Split video after Delta_save frames
                if frame_counter >= Delta_save:
                    out.release()
                    print(f"[INFO] Released movie_{movie_id}.mp4 with {frame_counter} frames.")
                    movie_id += 1
                    output_file = os.path.join(dir_experiment_run, 'movie', f'movie_{movie_id}.mp4')
                    out = cv2.VideoWriter(output_file, fourcc, fps, (frame_width, frame_height), isColor=is_color)
                    print(f"[INFO] Started new file: {output_file}")
                    frame_counter = 0
    
            time.sleep(0.001)
    
        if out:
            out.release()
            print(f"[INFO] Final video movie_{movie_id}.mp4 saved.")

else:
    def run_saving_img(queue_images,stop_saving,dir_experiment_run):
        if not os.path.exists(os.path.join(dir_experiment_run,'movie')):
            os.mkdir(os.path.join(dir_experiment_run,'movie'))
        Delta_save=int(fps*2*60)#video cropped in ~1minutes
        Dt_video = Delta_save*60 #sec
        x_movie, y_movie, t_movie = img_crop_ds.shape[0], img_crop_ds.shape[1], Delta_save
        movie_0=np.empty((x_movie, y_movie, t_movie),dtype='uint8')
        movie_1=np.empty((x_movie, y_movie, t_movie),dtype='uint8')
        movie_id_pari=True
        movie_id=0

        while not (stop_saving.is_set() and queue_images.empty()):
            if not queue_images.empty():
                frame,frame_id=queue_images.get()    
                frame_id=frame_id-movie_id*Delta_save
                if movie_id_pari:
                    movie_0[:,:,frame_id]=frame
                    if frame_id % (Delta_save-1) == 0 and frame_id!=0:
                        movie_id_pari=not movie_id_pari
                        np.save(os.path.join(dir_experiment_run,'movie',f'movie_{movie_id}'),movie_0)    
                        movie_id+=1
                        movie_0=np.empty((x_movie, y_movie, t_movie),dtype='uint8')
                else:
                    movie_1[:,:,frame_id]=frame
                    if frame_id % (Delta_save-1) == 0 and frame_id!=0:
                        movie_id_pari=not movie_id_pari
                        np.save(os.path.join(dir_experiment_run,'movie',f'movie_{movie_id}'),movie_1)    
                        movie_id+=1
                        movie_1=np.empty((x_movie, y_movie, t_movie),dtype='uint8')
            time.sleep(0.001) 
        if movie_id_pari:
            np.save(os.path.join(dir_experiment_run,'movie',f'movie_{movie_id}'),movie_0)  
        else:
            np.save(os.path.join(dir_experiment_run,'movie',f'movie_{movie_id}'),movie_1) 

# Start exp

In [ ]:
flag_plot, flag_img_bw, flag_save, static_stones, stones_bg_frame = True, False, True, True, False

In [ ]:
if hdf5_file:
    print("file is open")
    hdf5_file.close()
else:
    print("file is closed")

In [ ]:
# Get user input for experiment parameters
run_n = int(input("Enter run_n: ")) 

In [ ]:
# Set up experiment directory structure and file paths
dir_experiment_run=os.path.join(dir_experiment_fish,'run'+ str(run_n))
if not os.path.exists(os.path.join(path,dir_experiment_run)):
    print('OK: ',dir_experiment_run)
    os.mkdir(os.path.join(path,dir_experiment_run))
else:
    print('ERROR: want to overwrite the run?')

In [ ]:
# Create HDF5 file and datasets to save experimental data (positions, timestamps, etc.)
h5_experiment_run=os.path.join(dir_experiment_run,'run'+ str(run_n)+'.h5')
hdf5_file = h5py.File(h5_experiment_run, "w")

In [ ]:
#PARAMETERS
time_experiment=2*60*60#[s]-> 2h 60*60
real_time_exp=0

dl=8

shift_per_minut = 1 #1cm/min #to shift background stones
shift_per_second = 0.01*0.5/80 #calibrated 030125
shift2mm_per_second = 2*shift_per_second #calibrated 030125
shift2mm_per_second = shift_per_second/5 #calibrated 030125

In [ ]:
# Set up multi-threaded architecture for parallel camera capture, tracking, and motor control

global failed
failed=False

#Thred camera 
cameratimes=[]
cameratimestamp=[]
fromcamera=[0,np.zeros_like(bg_raw_crop_ds),1] #(timestamp, img_live, flag=0/1(for syncr.)) 
lock_fromcamera=threading.Lock() 
stop_camera=threading.Event()
trd_run_camera=threading.Thread(target=run_camera_simple,args=(lock_fromcamera, fromcamera, stop_camera))

#Thred plot
plottingtimes=[]
data_plotting=[img_crop_ds,np.array([150 , 150]),np.array([150 , 150]),np.array([150 , 150])] #(img_live, pos_live, flag=True/False to control plotting framerate)  
lock_plotting=threading.Lock() 
stop_plotting=threading.Event()
trd_run_plotting=threading.Thread(target=run_plotting,args=(lock_plotting, data_plotting, stop_plotting))

#Thred motor
motortimes=[]
motorpositions=[]
motor_live=[200,200,False,1] #(mot_x, mot_y, edge=True/False, flag_motor=1/0(for syncr.))  
tomotor=[xROIcenter,yROIcenter,False,1] #(fish_x, fish_y, fish_lost=True/False, flag=0/1(for syncr.))   
lock_tomotor=threading.Lock() 
lock_pos_motor=threading.Lock() 
stop_motor=threading.Event()
trd_run_motor=threading.Thread(target=run_motor,args=(stop_motor,tomotor,lock_tomotor,motor_live,lock_pos_motor)) #create thread to run the motor
 
#Thred tracking
trackingtimes=[]
pos=[]
frame_fish_yolo=[]
pos_fish_yolo=[]
pos_fish_kine=[]
conf_level=[]
head_dir=[]
debug_crop=[]
motor_pos_live=[]
motor_time_live=[]
fromtracking=[xROIcenter,yROIcenter,False,1] #(fish_x, fish_y, fish_lost=True/False,flag=0/1(for syncr.))  
lock_fromtracking=threading.Lock() 
stop_tracking=threading.Event()
trd_run_tracking=threading.Thread(target=run_tracking,args=(lock_fromcamera, fromcamera, lock_fromtracking, fromtracking, stop_tracking, lock_plotting, data_plotting, lock_pos_motor, motor_live))

if flag_save:#Thred saving images
    queue_images=queue.Queue() #queue to save the images
    stop_saving=threading.Event()  
    trd_save_images=threading.Thread(target=run_saving_img,args=(queue_images,stop_saving,dir_experiment_run))

In [ ]:
%matplotlib widget

In [ ]:
if flag_plot:
    fig,ax = plt.subplots(dpi=100)
    plot_live = ax.imshow(bg_mean,origin='lower',vmin=0,vmax=150,animated=True)
    (plot_kine,) = ax.plot(150, 150, 'go',animated=True)
    (plot_yolo,) = ax.plot(150, 150, 'yo',animated=True)
    (plot_pos,) = ax.plot(150, 150, 'ro',animated=True)
    bg=fig.canvas.copy_from_bbox(fig.bbox)
    fig.canvas.blit(fig.bbox)
    time.sleep(0.1)
    plt.show(block=False)

In [ ]:
_initialize(camera)

In [ ]:
_cleanup(camera)

In [ ]:
# Start with 'manual mode', and wait for 's' key press to proceed with automatic tracking
manual_interrupted=False
win=visual.Window(size=mon.getSizePix(),monitor=mon,fullscr=True,screen=screen_id,units='cm')
[img_static, rotated_im_array, im_pos, im_size]=closeloop_bg_screen(path, name_img, win)
print('manual mode')
manual_mode_screen(ser, int(x_center_motor), int(y_center_motor), win, dl, img_static)

date_i = time.asctime()
print('---💪Start experiment',date_i)
t0=time.perf_counter()
_initialize(camera)
trd_run_camera.start()
trd_run_tracking.start() 
trd_run_motor.start()

try:
    while real_time_exp < time_experiment:
        key=event.getKeys()

        phi0=img_static.phase
        phi1=phi0#+np.array([0, shift2mm_per_second])
        img_static.phase=phi1
        img_static.draw()
        win.flip()
    
        lock_fromtracking.acquire() #get centroids from camera thread
        x_centroid=fromtracking[0]
        y_centroid=fromtracking[1]
        fish_lost=fromtracking[2]
        flag_syncr=fromtracking[3]
        lock_fromtracking.release()
        
        time.sleep(0.001) #0.001
        
        lock_tomotor.acquire() #send centroids to motor thread
        tomotor[0]=x_centroid
        tomotor[1]=y_centroid 
        tomotor[2]=fish_lost
        tomotor[3]=flag_syncr
        lock_tomotor.release()
        
        time.sleep(0.001) #0.001
        
        lock_fromcamera.acquire() 
        real_time_exp=fromcamera[0]
        lock_fromcamera.release()

        if failed==True or key==['s']: 
            if key==['s']: 
                manual_interrupted=True
            event.clearEvents()  
            break  
except:
    print('ATT! something did not worked')
        
stop_motor.set()  #event to stop the motor
trd_run_motor.join() #wait for the motor thread to finish

stop_camera.set()  #event to stop the camera
trd_run_camera.join() #wait for the camera thread to finish

stop_tracking.set()  #event to stop the camera
trd_run_tracking.join() #wait for the camera thread to finish

if flag_plot:
    stop_plotting.set()  #event to stop the plotting
    trd_run_plotting.join() #wait for the plotting thread to finish
    
if flag_save:
    stop_saving.set() 
    trd_save_images.join() #  wait for the thread that is saving the pictures to finish 

if flag_img_bw:
    stop_saving_bw.set() 
    trd_save_images_bw.join()
    
date_f = time.asctime()

#saving
fish_id = hdf5_file.create_group("fish_id")
fish_id.attrs["description"] = "Define the fish identity"
fish_id.create_dataset("age", data=[age])
fish_id.create_dataset("sb", data=[sb])
fish_id.create_dataset("date_in_arena", data=[str(date_in_arena)])

exp = hdf5_file.create_group("exp")
exp.attrs["description"] = "Parameters of the experiment"
exp.create_dataset("fps", data=[fps])
exp.create_dataset("mm_per_pix", data=[round(mm_per_pix,3)])
exp.create_dataset("arena_border", data=[np.array([mot_xA,mot_yA,mot_xB,mot_yB,mot_xC,mot_yC,mot_xD,mot_yD])])
exp.create_dataset("agar_border", data=[agar_abs])
exp.create_dataset("stones_pos", data=[im_pos])
exp.create_dataset("stones_size", data=[im_size])
exp.create_dataset("gaussian_sigma", data=[gaussian_sigma])
exp.create_dataset("downsampling", data=[downsampling])

data = hdf5_file.create_group("data")
data.attrs["description"] = "Raw data coming from the experiment"
data.create_dataset("cameratimes", data=np.array(cameratimes))
data.create_dataset("cameratimestamp", data=np.array(cameratimestamp))
data.create_dataset("pos_fish_yolo", data=np.array(pos_fish_yolo))
data.create_dataset("frame_fish_yolo", data=np.array(frame_fish_yolo))
data.create_dataset("motorpositions", data=np.array(motorpositions))
data.create_dataset("motortimes", data=np.array(motortimes))
data.create_dataset("trackingtimes", data=np.array(trackingtimes))
data.create_dataset("date_i_run", data=[str(date_i)])
data.create_dataset("date_f_run", data=[str(date_f)])

#closing
print('---✔️End experiment')
print('Close win and camera.')
hdf5_file.close()
win.close()
_cleanup(camera)

## control time profile

In [ ]:
%matplotlib widget
cameratimes=np.array(cameratimes)
cameratimestamp=np.array(cameratimestamp)
trackingtimes=np.array(trackingtimes)
plottingtimes=np.array(plottingtimes)
motortimes=np.array(motortimes)

fig,axs=plt.subplots(4, 1, figsize=(7,8))
axs[0].plot(cameratimes[:-1]-cameratimes[0],np.diff(cameratimes), '.-', alpha=0.3, label='t0='+str(round(cameratimes[0],2))+'\ntn='+str(round(cameratimes[-1],2)))
axs[0].plot(cameratimestamp[:-1]-cameratimestamp[0],np.diff(cameratimestamp), 'k.', alpha=0.5)
axs[0].set_xlabel('time experiment (s)')
axs[0].legend()
axs[0].xaxis.set_visible(False)
axs[0].set_ylabel('camera fr (s)')
axs[0].set_title('mean='+str(round(1/np.mean(np.diff(cameratimes))))+'Hz')

axs[1].plot(trackingtimes[:-1]-trackingtimes[0],np.diff(trackingtimes), '.-', alpha=0.3, label='t0='+str(round(trackingtimes[0],2))+'\ntn='+str(round(trackingtimes[-1],2)))
axs[1].set_xlabel('time experiment (s)')
axs[1].set_ylabel('tracking fr (s)')
axs[1].xaxis.set_visible(False)
axs[1].legend()
axs[1].set_title('mean='+str(round(1/np.mean(np.diff(trackingtimes))))+'Hz')

if flag_plot:
    axs[2].plot(plottingtimes[:-1]-plottingtimes[0],np.diff(plottingtimes), '.-', alpha=0.3, label='t0='+str(round(plottingtimes[0],2))+'\ntn='+str(round(plottingtimes[-1],2)))
    axs[2].set_title('mean='+str(round(1/np.mean(np.diff(plottingtimes))))+'Hz')
    axs[2].legend()
axs[2].set_xlabel('time experiment (s)')
axs[2].set_ylabel('plotting fr (s)')
axs[2].xaxis.set_visible(False)

axs[3].plot(motortimes[:-1]-motortimes[0],np.diff(motortimes), '.-', alpha=0.3, label='t0='+str(round(motortimes[0],2))+'\ntn='+str(round(motortimes[-1],2)))
axs[3].set_xlabel('time experiment (s)')
axs[3].set_ylabel('motor fr (s)')
axs[3].legend()
axs[3].set_title('mean='+str(round(1/np.mean(np.diff(motortimes))))+'Hz')

print('Starting time:\n motor=',motortimes[0],'\n tracking=',trackingtimes[0],'\n camera(movie)=',cameratimes[0])
fig.suptitle('Tot time:'+str(round(motortimes[-1]/60,0))+'min')
plt.show()

In [ ]:
## control saving

In [ ]:
print(dir_experiment_run)
n_video=0
for file in os.listdir(os.path.join(dir_experiment_run,'movie')):
    print(file)
    n_video+=1
print('Tot. number of video:', n_video)

In [ ]:
if video_mp4:
    for file in os.listdir(os.path.join(dir_experiment_run, 'movie')):
        print('video is saved in .mp4')
        if file.startswith('movie_0') and file.endswith('.mp4'):
            video_path = os.path.join(dir_experiment_run, 'movie', file)
            myVideo = load_mp4_as_numpy(video_path, as_gray=True)  # or as_gray=False for RGB
            print(f"Loaded video with shape: {myVideo.shape}")
else:
    for file in os.listdir(os.path.join(dir_experiment_run, 'movie')):
        if file.startswith('movie_0') and file.endswith('.npy'):
            myVideo = np.load(os.path.join(dir_experiment_run, 'movie', file))
            print(f"Loaded video with shape: {myVideo.shape}")

In [ ]:
%matplotlib inline
def slider_movie(frame):
    fig,ax = plt.subplots(figsize=(5,5), dpi=100)
    ax.imshow(myVideo[:,:,frame],origin='lower',cmap='magma')
interact(slider_movie,frame=IntSlider(value=200,min=0,max=myVideo[:,:,:].shape[2]-1,step=1))

## control traj

In [ ]:
hdf5_file.close()

In [ ]:
h5_experiment_run

In [ ]:
hdf5_file = h5py.File(h5_experiment_run, "r")
h5tree_view(hdf5_file)

In [ ]:
cameratimes=hdf5_file['data/cameratimes'][:]
cameratimestamp=hdf5_file['data/cameratimestamp'][:]
centroid_tot=hdf5_file['data/pos_fish_yolo'][:]
motorpositions=hdf5_file['data/motorpositions'][:]
motortimes=hdf5_file['data/motortimes'][:]
trackingtimes=hdf5_file['data/trackingtimes'][:]

In [ ]:
print(hdf5_file['exp/mm_per_pix'][:])

In [ ]:
#--1.time
time_traj = int(motortimes[-1]-motortimes[0])
l=250*time_traj
print(time_traj,time_traj/l)
print('Is it exactly 250Hz?',1/(time_traj/l))
tvals = np.linspace(0, time_traj, l)

In [ ]:
#--2.x_f: online -> pos.
pos_mm=np.array(centroid_tot)*mm_per_pix
xinterp_centroid=np.interp(tvals, trackingtimes,pos_mm[:-1,0]) #IMP: the camera is on even if the last video fail to be saved (fish lost before 1min), so you need to select the fisrt [:myVideo.shape[2]] points
yinterp_centroid=np.interp(tvals, trackingtimes,pos_mm[:-1,1])

In [ ]:
print(trackingtimes.shape,tvals.shape,pos_mm.shape)

In [ ]:
#--3.X_M
motortimes_shift=motortimes#-0.04
xinterp_motor=np.interp(tvals, motortimes_shift,motorpositions[:,0])
yinterp_motor=np.interp(tvals, motortimes_shift,motorpositions[:,1])

In [ ]:
#--4.Reconstruction: X_lab = x_f + X_M
X=xinterp_motor#+xinterp_centroid
Y=yinterp_motor#+yinterp_centroid

In [ ]:
fig, ax = plt.subplots(figsize=(3,3),dpi=200)
#ax.imshow(new_im_array[:,::-1],origin='lower',alpha=1,extent=[x_center_motor-im_pos[0]-im_size[0]/2,x_center_motor-im_pos[0]+im_size[0]/2,y_center_motor-im_pos[1]-im_size[1]/2,y_center_motor-im_pos[1]+im_size[1]/2])
#ax.plot([xA_abs,xB_abs,xC_abs,xD_abs,xA_abs],[yA_abs,yB_abs,yC_abs,yC_abs,yA_abs], 'k-' )
ax.plot(motorpositions[:,0],motorpositions[:,1],'k',alpha=0.3,label='motor position')
ax.plot(X, Y,'r-',ms=1,alpha=0.3,label='fish position')
ax.plot(X[0], Y[0],'yo',label='start')
ax.axis('equal')
#plt.legend()
plt.show()

# Plot all fish

In [ ]:
print(dir_experiment_fish)
n_run_tot=0
for file in os.listdir(dir_experiment_fish):
    if file[0:3]=='run':
        n_run_tot=n_run_tot+1
tot_traj=[[] for i in range(n_run_tot)]
tot_times=[[] for i in range(n_run_tot)]
category_names=[[] for i in range(n_run_tot)]
print(n_run_tot)
for i in range(n_run_tot):
    h5_experiment_run=os.path.join(dir_experiment_fish,'run'+str(i+1),'run'+str(i+1)+'.h5')
    print(h5_experiment_run)
    hdf5_file = h5py.File(h5_experiment_run, "r")
    try:
        tot_traj[i]=hdf5_file['data/motorpositions'][:]
        tot_times[i]=round(hdf5_file['data/motortimes'][-1]/60,1)
        category_names[i]='run'+str(i+1)
    except:
        tot_times[i]=0
        tot_traj[i]=np.array([[     207.08,      40.633]])
        category_names[i]='run'+str(i+1)
        n_run_tot=n_run_tot-1
print(n_run_tot)

In [ ]:
results = {
    'time\n(min)': tot_times,
}
d = datetime.datetime.now()
only_date, only_time = d.date(), d.time()
print(only_date)

title=str(only_date)+'_fish'+str(fish_number)+'_'+str(age)+'dpf'

def survey(results, category_names, title):
    """
    Parameters
    ----------
    results : dict
        A mapping from question labels to a list of answers per category.
        It is assumed all lists contain the same number of entries and that
        it matches the length of *category_names*.
    category_names : list of str
        The category labels.
    """
    labels = list(results.keys())
    data = np.array(list(results.values()))
    data_cum = data.cumsum(axis=1)
    category_colors = plt.colormaps['tab10']( #RdYlGn
        np.linspace(0.15, 0.85, data.shape[1]))

    fig, ax = plt.subplots(2,1,figsize=(5, 5),gridspec_kw={'height_ratios': [1, 6]})
    ax[0].invert_yaxis()
    ax[0].xaxis.set_visible(False)
    ax[0].set_xlim(0, np.sum(data, axis=1).max())

    for i, (colname, color) in enumerate(zip(category_names, category_colors)):
        widths = data[:, i]
        starts = data_cum[:, i] - widths
        rects = ax[0].barh(labels, widths, left=starts, height=0.5,
                        label=colname, color=color)

        r, g, b, _ = color
        text_color = 'white' if r * g * b < 0.5 else 'darkgrey'
        ax[0].bar_label(rects, label_type='center', color=text_color)
        ax[0].set_title(title)
           # ax[0].legend(ncols=len(category_names), bbox_to_anchor=(0, 1),loc='lower left', fontsize='small')
    return fig, ax, category_colors
[fig,ax,category_colors]=survey(results, category_names,title)

#ax[1].imshow(new_im_array[:,::-1],origin='lower',alpha=0.3,extent=[x_center_motor-im_pos[0]-im_size[0]/2,x_center_motor-im_pos[0]+im_size[0]/2,y_center_motor-im_pos[1]-im_size[1]/2,y_center_motor-im_pos[1]+im_size[1]/2])
for i in range(n_run_tot):
    ax[1].plot(tot_traj[i][:,0],tot_traj[i][:,1],c=category_colors[i],alpha=1)
    ax[1].set_xlabel('X (mm)')
    ax[1].set_ylabel('Y (mm)')
    ax[1].set_xlim(0,400)
    ax[1].set_ylim(0,400)
   # ax[1].axis('equal')

plt.savefig(os.path.join(dir_experiment_fish,title+".svg"), dpi=150)
plt.savefig(os.path.join(dir_experiment_fish,title+".jpg"), dpi=150)
plt.show()